# [6.4] Crosscoders and Model Diffing - Exercises

Crosscoders compare two models by decomposing paired activations into shared and model-specific parts. In this notebook you build the validation ladder on deterministic tensors: reconstruction, feature ownership, behavior-delta prediction, paired score deltas, and a target-vs-random control.

```yaml
gt_tier: GT-3 paired-model diffing preflight
exercise_id: 6_4_crosscoders_and_model_diffing
expected_runtime: 60-90 minutes for CPU exercises; several minutes for CUDA paired-model preflight
requires_gpu: true for the TransformerLens preflight; false for the implementation exercises
```

<details>
<summary>Expected output</summary>

By the end, each local test should print an "All tests ... passed" line. The final report-backed cells should show pinned `gelu-1l` and `solu-1l` metrics: exact shared-plus-delta reconstruction, a strong top SVD model-diff direction, technical-vs-everyday label separation, and a top-direction removal control that beats an orthogonal random direction.

</details>

<details>
<summary>Help - how to read this section</summary>

Treat each result as a filter. Reconstruction proves the paired activation bookkeeping. Specificity suggests ownership. Behavior-delta prediction checks whether a direction tracks a paired label. Direction-removal controls decide whether the direction is stronger than a matched random alternative.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part4_crosscoders_model_diffing"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_crosscoders_model_diffing.tests as tests

FeatureOwner = Literal["shared", "model_a", "model_b"]


@dataclass(frozen=True)
class CrosscoderOutput:
    shared_acts: t.Tensor
    model_a_specific_acts: t.Tensor
    model_b_specific_acts: t.Tensor
    reconstructed_model_a: t.Tensor
    reconstructed_model_b: t.Tensor


@dataclass(frozen=True)
class CrosscoderReconstructionReport:
    model_a_mse: float
    model_b_mse: float
    shared_active_fraction: float
    model_a_passes: bool
    model_b_passes: bool
    shared_reconstructs_both: bool


@dataclass(frozen=True)
class FeatureSpecificityReport:
    feature_id: int
    model_a_mean: float
    model_b_mean: float
    specificity: float
    owner: FeatureOwner


@dataclass(frozen=True)
class BehaviorDeltaPredictionReport:
    feature_id: int
    auc: float
    positive_mean: float
    negative_mean: float
    passes_threshold: bool


@dataclass(frozen=True)
class CrosscoderAblationReport:
    baseline_delta: float
    ablated_delta: float
    random_ablated_delta: float
    delta_reduction: float
    random_reduction: float
    passes_control: bool


## 1. Crosscoder Reconstruction

The crosscoder has shared latents and model-specific latents. Shared latents decode into both model spaces; model-specific latents decode only into their matching model.

<details>
<summary>Expected output</summary>

The controlled example reconstructs model A as `[[3.0, 0.0], [3.0, 1.0]]`, model B as `[[1.0, 4.0], [0.0, 6.0]]`, and the reconstruction report passes for both spaces.

```text
All tests in `test_decode_crosscoder_reconstructs_shared_and_specific_spaces` passed!
```

</details>

<details>
<summary>Help - shared is not copied twice</summary>

Shared features can decode differently into the two model spaces. The constraint is participation in both reconstructions, not identical decoder weights.

</details>

Common bug: decoding model-A-specific latents into model B as well, which hides leakage until the specific decoders are asymmetric.


In [ ]:
def decode_crosscoder(
    shared_acts: t.Tensor,
    model_a_specific_acts: t.Tensor,
    model_b_specific_acts: t.Tensor,
    shared_decoder_a: t.Tensor,
    shared_decoder_b: t.Tensor,
    model_a_decoder: t.Tensor,
    model_b_decoder: t.Tensor,
) -> CrosscoderOutput:
    raise NotImplementedError()


def crosscoder_reconstruction_report(
    model_a_activations: t.Tensor,
    model_b_activations: t.Tensor,
    output: CrosscoderOutput,
    *,
    mse_threshold: float = 1e-6,
) -> CrosscoderReconstructionReport:
    raise NotImplementedError()


tests.test_decode_crosscoder_reconstructs_shared_and_specific_spaces(
    decode_crosscoder,
    crosscoder_reconstruction_report,
)


## 2. Feature Specificity

Classify each feature by comparing its mean activation in model A and model B. Differences within `shared_threshold` count as shared.

<details>
<summary>Expected output</summary>

The controlled tensor should classify features as `['shared', 'model_a', 'model_b']`, and feature 2 should be model-B-specific.

```text
All tests in `test_feature_specificity_classifies_shared_model_a_and_model_b` passed!
```

</details>

<details>
<summary>Help - ownership has a sign</summary>

Specificity is `model_b_mean - model_a_mean`. Absolute differences can tell you that a feature differs, but they erase which model uses the feature more.

</details>

Common bug: using absolute specificity for ownership and then being unable to distinguish model-A-specific from model-B-specific features.


In [ ]:
def feature_specificity_report(
    model_a_feature_acts: t.Tensor,
    model_b_feature_acts: t.Tensor,
    feature_id: int,
    *,
    shared_threshold: float = 0.1,
) -> FeatureSpecificityReport:
    raise NotImplementedError()


def classify_features_by_specificity(
    model_a_feature_acts: t.Tensor,
    model_b_feature_acts: t.Tensor,
    *,
    shared_threshold: float = 0.1,
) -> list[FeatureOwner]:
    raise NotImplementedError()


tests.test_feature_specificity_classifies_shared_model_a_and_model_b(
    feature_specificity_report,
    classify_features_by_specificity,
)


## 3. Behavior-Delta Prediction

If a model-specific direction explains a model difference, it should predict which paired examples show that behavior delta. Use signed AUC so predictive and antipredictive directions are both visible.

<details>
<summary>Expected output</summary>

The toy feature has signed AUC `1.0`, an inverted feature has raw AUC `0.0` but signed AUC `1.0`, and the report passes the threshold.

```text
All tests in `test_behavior_delta_prediction_uses_signed_auc_and_means` passed!
```

</details>

<details>
<summary>Help - signed AUC keeps opposite directions visible</summary>

A direction can be useful because it is high on positives or low on positives. Signed AUC lets either ranking pass, while positive and negative means tell you which direction you found.

</details>

Common bug: reporting accuracy at one arbitrary threshold instead of checking the rank ordering across thresholds.


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def behavior_delta_prediction_report(
    feature_scores: t.Tensor,
    behavior_delta_labels: t.Tensor,
    *,
    feature_id: int,
    min_auc: float = 0.8,
) -> BehaviorDeltaPredictionReport:
    raise NotImplementedError()


tests.test_behavior_delta_prediction_uses_signed_auc_and_means(
    behavior_delta_prediction_report,
    roc_auc_binary,
)


## 4. Paired Behavior Deltas

Paired score deltas must specify direction. In this section, a behavior delta is model B minus model A.

<details>
<summary>Expected output</summary>

Model A scores `[0.25, 0.5]` and model B scores `[0.75, 0.25]` should produce `[0.5, -0.25]`.

```text
All tests in `test_toy_behavior_delta_scores_are_model_b_minus_model_a` passed!
```

</details>

<details>
<summary>Help - delta direction is part of the claim</summary>

The sign says whether model B increased or decreased the behavior relative to model A. Taking absolute values too early makes the model difference uninterpretable.

</details>

Common bug: computing `abs(model_b - model_a)` and then trying to reason about which model changed.


In [ ]:
def toy_behavior_delta_scores(
    model_a_scores: t.Tensor,
    model_b_scores: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_toy_behavior_delta_scores_are_model_b_minus_model_a(
    toy_behavior_delta_scores,
)


## 5. Direction-Removal Controls

Prediction is weaker than intervention. The target feature should reduce behavior-delta magnitude more than a random matched feature.

<details>
<summary>Expected output</summary>

Target ablation should reduce the behavior delta by `0.75`, random ablation by `0.15`, and the test should fail when those roles are swapped.

```text
All tests in `test_crosscoder_ablation_requires_target_to_beat_random_control` passed!
```

</details>

<details>
<summary>Help - prediction is weaker than intervention</summary>

A feature can predict labels without explaining the model difference. The control asks whether removing the target feature reduces the delta more than removing a matched random feature.

</details>

Common bug: checking only that ablation changes the metric, rather than requiring a larger reduction than the control.


In [ ]:
def crosscoder_ablation_report(
    baseline_behavior_deltas: t.Tensor,
    ablated_behavior_deltas: t.Tensor,
    random_ablated_behavior_deltas: t.Tensor,
) -> CrosscoderAblationReport:
    raise NotImplementedError()


tests.test_crosscoder_ablation_requires_target_to_beat_random_control(
    crosscoder_ablation_report,
)


## Whole-Notebook Contract

Once all exercises pass, your implementation should satisfy the same smoke-test contract as `solutions.py`.

<details>
<summary>Expected output</summary>

After uncommenting the last line, the test should print:

```text
All tests in `test_notebook_contract` passed!
```

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    shared = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    model_a_specific = t.tensor([[2.0], [3.0]])
    model_b_specific = t.tensor([[4.0], [5.0]])
    shared_decoder = t.eye(2)
    output = decode_crosscoder(
        shared,
        model_a_specific,
        model_b_specific,
        shared_decoder,
        shared_decoder,
        t.tensor([[1.0, 0.0]]),
        t.tensor([[0.0, 1.0]]),
    )
    target_a = t.tensor([[3.0, 0.0], [3.0, 1.0]])
    target_b = t.tensor([[1.0, 4.0], [0.0, 6.0]])
    model_a_features = t.tensor([[1.0, 3.0, 0.1], [1.0, 2.0, 0.2]])
    model_b_features = t.tensor([[1.1, 0.2, 4.0], [1.0, 0.1, 5.0]])
    return {
        "reconstruction": {
            "reconstructed_model_a": output.reconstructed_model_a.tolist(),
            "reconstructed_model_b": output.reconstructed_model_b.tolist(),
            "report": crosscoder_reconstruction_report(target_a, target_b, output).__dict__,
        },
        "specificity": {
            "owners": classify_features_by_specificity(
                model_a_features,
                model_b_features,
                shared_threshold=0.2,
            ),
            "feature_2": feature_specificity_report(
                model_a_features,
                model_b_features,
                2,
                shared_threshold=0.2,
            ).__dict__,
        },
        "behavior_delta": behavior_delta_prediction_report(
            t.tensor([0.1, 0.2, 0.9, 1.0]),
            t.tensor([0, 0, 1, 1], dtype=t.bool),
            feature_id=7,
        ).__dict__,
        "ablation": crosscoder_ablation_report(
            t.tensor([1.0, 1.0]),
            t.tensor([0.2, 0.3]),
            t.tensor([0.8, 0.9]),
        ).__dict__,
        "delta_scores": {
            "behavior_deltas": toy_behavior_delta_scores(
                t.tensor([0.25, 0.5]),
                t.tensor([0.75, 0.25]),
            ).tolist(),
        },
    }


# Uncomment after finishing all exercises.
# tests.test_notebook_contract(run_smoke_test)


## Signature Result

The final result is report-backed and uses the committed CUDA evidence. It does not rerun the TransformerLens path inside the notebook; rerun `solutions.run_gpu_test(max_vram_gb=24.0)` from Python when you want to refresh the report.

<details>
<summary>Expected output</summary>

The table should show `gelu-1l` vs `solu-1l`, 8 technical and 8 everyday prompts, activation shape `[16, 512]`, reconstruction MSE near zero, top variance fraction around `0.805`, technical/everyday label AUC `1.0`, direction-removal reduction around `16.084` versus random reduction around `0.005`, and peak VRAM under `1 GB`.

</details>

<details>
<summary>Interpreting the signature result</summary>

This proves a scoped paired-checkpoint preflight: exact shared-plus-delta reconstruction, a top residual-delta SVD direction that separates generated prompt labels, and a direction-removal control that beats an orthogonal random direction. It is not a trained crosscoder paper replication.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model A / B", f"{gpu['model_a_name']} / {gpu['model_b_name']}"),
        ("model A revision", gpu["model_a_revision"][:12]),
        ("model B revision", gpu["model_b_revision"][:12]),
        ("prompts", f"{gpu['technical_prompt_count']} technical / {gpu['everyday_prompt_count']} everyday"),
        ("activation shape", gpu["activation_shape"]),
        ("model A / B MSE", f"{gpu['model_a_mse']:.2e} / {gpu['model_b_mse']:.2e}"),
        ("shared active fraction", round(gpu["shared_active_fraction"], 3)),
        ("top singular value", round(gpu["top_singular_value"], 3)),
        ("top variance fraction", round(gpu["top_variance_fraction"], 3)),
        ("technical/everyday label AUC", round(gpu["behavior_delta_auc"], 3)),
        ("top-direction projection means", f"{gpu['behavior_delta_positive_mean']:.3f} / {gpu['behavior_delta_negative_mean']:.3f}"),
        ("baseline delta norm", round(gpu["baseline_delta_norm"], 3)),
        ("top-direction removed norm", round(gpu["top_direction_ablated_delta_norm"], 3)),
        ("orthogonal-control removed norm", round(gpu["random_direction_ablated_delta_norm"], 3)),
        ("activation-delta reduction", round(gpu["delta_reduction"], 3)),
        ("random-direction reduction", round(gpu["random_reduction"], 3)),
        ("floor-vs-top diagnostic abs mean", round(gpu["floor_top_delta_abs_mean"], 3)),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["model A", "model B"],
    [gpu["model_a_mse"], gpu["model_b_mse"]],
    color=["#2563eb", "#0f766e"],
)
axes[0].set_title("Exact reconstruction")
axes[0].set_ylabel("MSE, lower is better")

axes[1].bar(
    ["technical", "everyday"],
    [gpu["behavior_delta_positive_mean"], gpu["behavior_delta_negative_mean"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title(f"Top direction label AUC = {gpu['behavior_delta_auc']:.3f}")
axes[1].set_ylabel("projection mean")

axes[2].bar(
    ["baseline", "top removed", "orthogonal"],
    [
        gpu["baseline_delta_norm"],
        gpu["top_direction_ablated_delta_norm"],
        gpu["random_direction_ablated_delta_norm"],
    ],
    color=["#2563eb", "#16a34a", "#f97316"],
)
axes[2].set_title("Direction-removal control")
axes[2].set_ylabel("activation-delta norm")
axes[2].tick_params(axis="x", rotation=15)

fig.tight_layout()
plt.show()


## Limitations

The local tests use tiny deterministic tensors. The CUDA report uses two small public TransformerLens checkpoints, 16 generated safe prompts, an exact shared-plus-delta construction, and an SVD direction over residual deltas. It does not prove learned model-specific sparse features, broad base-vs-instruction behavior, generated-completion changes, or causal control over model outputs.
